# Library Importation

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score


# Load Model & Dataset

In [2]:
# loading the pre-trained model (trained on IoT data)
model = joblib.load("carbon_predictor_model.pkl")
# loading the scaler that was fitted on IoT data during training
scaler = joblib.load("carbon_scaler.pkl")
# The primary data from survey
df_pri_raw = pd.read_csv("primary_data_CO2.csv")

# FEATURES
features = ["Energy_Usage_kWh", "Transportation_Distance_km", "Plastic_Usage_kg"]

# TRANSFORM THE PRIMARY DATA
# Apply the IoT-fitted scaler to survey data
# X_pri_scaled = scaler.transform(df_pri_raw[features])


# Local Validation

In [3]:
# Make prediction using the prepared primary data
y_pred_iot = model.predict(df_pri_raw[features])

# Actual CO2 values from primary survey data
y_actual = df_pri_raw["Carbon_Emission_kgCO2"]

# CALIBRATION LAYER
# The IoT model predicts higher than primary actuals due to stronger emission factors
# This step learns a correction factor to bring IoT predictions into the primary data range
calibrator = LinearRegression()
calibrator.fit(y_pred_iot.reshape(-1, 1), y_actual)
y_pred = calibrator.predict(y_pred_iot.reshape(-1, 1))

print(f"Calibration scale factor: {calibrator.coef_[0]:.4f}")
print(f"Calibration intercept:    {calibrator.intercept_:.4f}")


Calibration scale factor: 0.3995
Calibration intercept:    -0.0584


# Data Comparison

In [4]:
# CALCULATE MODEL PERFORMANCE
mae = mean_absolute_error(y_actual, y_pred)
r2 = r2_score(y_actual, y_pred)

print("--- Local Validation Results ---")
print(f"Mean Absolute Error (MAE): {mae:.2f} kg CO2")
print(f"R² Score: {r2:.4f}")

# Show the first 5 comparisons
comparison = pd.DataFrame({
    'Actual': y_actual,
    'Predicted': y_pred,
    'Difference': y_actual - y_pred
})

print("\nSample Comparisons:")
print(comparison.head())

--- Local Validation Results ---
Mean Absolute Error (MAE): 0.43 kg CO2
R² Score: 0.9368

Sample Comparisons:
     Actual  Predicted  Difference
0  1.569816   1.778732   -0.208916
1  4.631780   4.741840   -0.110061
2  5.477780   5.938795   -0.461015
3  5.282745   4.343474    0.939271
4  3.718695   3.346444    0.372252
